# 20 Batch runner

Build a concrete run sequence for one preset, with explicit stop points and a PowerShell script template.

This notebook is **non-destructive**. It does not apply file changes. It only plans and exports a guided sequence.


In [ ]:
from pathlib import Path
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNNER_LOG_DIR = PROJECT_ROOT / 'data' / 'runner_logs'
RUNNER_LOG_DIR.mkdir(parents=True, exist_ok=True)

from src.batch_presets import list_presets
from src.batch_runner import (
    RunnerConfig,
    build_batch_sequence,
    summarize_batch_sequence,
    build_powershell_commands,
    export_batch_runner_tables,
)


In [ ]:
PRESET_NAME = 'review_only'  # review_only, canonicalize_unresolved, ocr_rescue, promote_to_execution, apply_small_live_batch, rollback_last_batch
STOP_AFTER_REVIEW = True
STOP_AFTER_MANIFEST = True
STOP_BEFORE_APPLY = True
STOP_BEFORE_ROLLBACK = True
INCLUDE_OPTIONAL_DONE_STEPS = False
ALLOW_LIVE_APPLY = False
RUNNER_NOTES = 'Default safe guided runner.'

cfg = RunnerConfig(
    preset_name=PRESET_NAME,
    stop_after_review=STOP_AFTER_REVIEW,
    stop_after_manifest=STOP_AFTER_MANIFEST,
    stop_before_apply=STOP_BEFORE_APPLY,
    stop_before_rollback=STOP_BEFORE_ROLLBACK,
    include_optional_done_steps=INCLUDE_OPTIONAL_DONE_STEPS,
    allow_live_apply=ALLOW_LIVE_APPLY,
    notes=RUNNER_NOTES,
)


In [ ]:
display(list_presets()[['name', 'description', 'goal']])


,name,description,goal
0,review_only,Run the safe read-only core pipeline up to rev...,"Inspect current files, duplicates, junk, extra..."
1,canonicalize_unresolved,Run the quality branch to canonicalize unresol...,Improve deterministic canonical paths and name...
2,ocr_rescue,Run the OCR branch for scanned PDFs and image-...,Rescue weak/no-text files so content-derived f...
3,promote_to_execution,Promote newly canonical-ready rows back into p...,Convert accepted deterministic results into pl...
4,apply_small_live_batch,Prepare and execute a small approved batch wit...,"Run a tiny live batch only after dry-run, revi..."
5,rollback_last_batch,Rollback the most recent moved batch and then ...,Reverse the last applied batch and verify obse...


In [ ]:
sequence = build_batch_sequence(OUTPUT_DIR, cfg)
summary = summarize_batch_sequence(sequence, cfg)
summary


,preset_name,description,goal,steps_total,steps_pending,steps_done,stop_points,next_run_notebook,first_stop_notebook,allow_live_apply,notes
0,review_only,Run the safe read-only core pipeline up to rev...,"Inspect current files, duplicates, junk, extra...",7,0,7,2,None,05_review_outputs.ipynb,False,Default safe guided runner.


In [ ]:
display(sequence[['run_order', 'stage', 'notebook', 'status', 'runner_action', 'stop_point', 'stop_reason']])


,run_order,stage,notebook,status,runner_action,stop_point,stop_reason
0,1,01_policy_check,01_policy_check.ipynb,done,already_done,False,
1,2,02_inventory,02_inventory.ipynb,done,already_done,False,
2,3,03_rule_classification,03_rule_classification.ipynb,done,already_done,False,
3,4,04_extract_text,04_extract_text.ipynb,done,already_done,False,
4,5,05_review_outputs,05_review_outputs.ipynb,done,stop_checkpoint,True,Review checkpoint before planning or any execu...
5,6,06_planner,06_planner.ipynb,done,already_done,False,
6,7,07_execution_manifest,07_execution_manifest.ipynb,done,stop_checkpoint,True,Manifest checkpoint before any apply action.


In [6]:
ps_lines = build_powershell_commands(PROJECT_ROOT, sequence)
ps_path = RUNNER_LOG_DIR / 'batch_runner_run_latest.ps1'
ps_path.write_text('\n'.join(ps_lines), encoding='utf-8')

summary_path, sequence_path, suggested_ps_path = export_batch_runner_tables(summary, sequence, OUTPUT_DIR)
print('Summary export:', summary_path)
print('Sequence export:', sequence_path)
print('PowerShell script:', ps_path)

Summary export: C:\00_Developement\sch-file-organizer\data\outputs\batch_runner_summary_latest.csv
Sequence export: C:\00_Developement\sch-file-organizer\data\outputs\batch_runner_sequence_latest.csv
PowerShell script: C:\00_Developement\sch-file-organizer\data\runner_logs\batch_runner_run_latest.ps1


## How to use

- Keep `ALLOW_LIVE_APPLY = False` unless you have already reviewed manifests and want a tiny approved live batch.
- The PowerShell script uses `jupyter nbconvert --execute --inplace` to run notebooks in sequence with pause prompts at stop points.
- For normal day-to-day use, this notebook is mainly a **guided controller**: review the sequence, then run the next notebook when you are ready.
